# Lab 2: Hugging Face Transformers — Encoder vs Decoder

Goal: Learn to use the `transformers` library to work with models, both encoder-only (e.g., BERT) and decoder-only (e.g., Gemma 1B).

In this lab we will cover:
- Overview of the library and model types
- Import and use of BERT, focusing on `cache_dir` and loading/inference parameters
- Use of a Gemma model, focusing on special tokens, tokenization, and chat template
- Two hands-on exercises: embedding similarity and how to construct effective prompts

## Step 0 - What is Huggingface

<style>
img[src="https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image4.png?raw=1"] {
    width: 40%;
    height: auto;
}
</style>
![alt text](https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image4.png?raw=1)


## Lab overview: encoder-only vs decoder-only

- Encoder-only models (e.g., BERT): produce text representations (embeddings). Ideal for classification, semantic search, and similarity.
- Decoder-only models (e.g., Gemma): generate text autoregressively and are suited for dialogue, completion, and extraction via prompting.

The `transformers` library offers uniform interfaces for tokenizers and models, supports caching with `cache_dir`, and includes utilities such as a chat template for conversation-trained models.

In [14]:
# Setup: import and config
import os, json
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)

# Main paths (local cache for models)
BASE_DIR = os.path.join('lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

Transformers version: 5.0.0


We will use `bert-base-uncased` to generate sentence embeddings. Focus:
- `cache_dir`: local directory to save weights/tokenizer
- Loading parameters: dtype, device map, etc. (here, CPU-only)
- Inference: obtain embeddings from `last_hidden_state` and compute cosine similarity


In [15]:
# Caricamento tokenizer e modello BERT
bert_model_id = 'bert-base-uncased' # Check https://huggingface.co/google-bert/bert-base-uncased
tokenizer_bert = AutoTokenizer.from_pretrained(bert_model_id, cache_dir=CACHE_DIR)
model_bert = AutoModel.from_pretrained(bert_model_id, cache_dir=CACHE_DIR)

print('Model structure:\n', model_bert, "\n\n")
print('The model has been loaded on memory using this data type:', next(model_bert.parameters()).dtype)
print('The model has been loaded on this device:', next(model_bert.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model structure:
 BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1,

The model is on cpu. It needs to be moved somewhere where computation is faster.

In [16]:
device = 'cuda'
model_bert.to(device)
print('The model has been loaded on this device:', next(model_bert.parameters()).device)

The model has been loaded on this device: cuda:0


Example on how to embed a sequence using BERT

In [17]:
# Example on how to embed a sequence using BERT
text = "Hello, how are you?"
inputs = tokenizer_bert(
    text,
    return_tensors='pt' # pt => pytorch tensor
).to(device)

# Indices associated to each input token.
print('Input indices: ', inputs)


# Define tokens' indices and attention mask. We need to pass them as input
# to the bert model.
example_input_ids = inputs['input_ids']
# Attention mask: set to 1 if bert should pay attention to the token, else 0.
# We set to zero the attention of padding tokens, for example.
example_attention_mask = inputs['attention_mask']

# Forward pass.
outputs = model_bert(example_input_ids, attention_mask=example_attention_mask)
# We're intereseted in last hidden state of bert (i.e. contextual embeddings).
embeddings = outputs.last_hidden_state

# Print info.
print('\nEmbeddings shape:', embeddings.shape)
print(f"\tBatch size:{embeddings.shape[0]}")
print(f"\tInput size :{embeddings.shape[1]}")
print(f"\tEmbedding size : {embeddings.shape[2]}")
print('\nEmbeddings:', embeddings)

Input indices:  {'input_ids': tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 1029,  102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Embeddings shape: torch.Size([1, 8, 768])
	Batch size:1
	Input size :8
	Embedding size : 768

Embeddings: tensor([[[-0.0824,  0.0667, -0.2880,  ..., -0.3566,  0.1960,  0.5381],
         [ 0.0310, -0.1448,  0.0952,  ..., -0.1560,  1.0151,  0.0947],
         [-0.8935,  0.3240,  0.4184,  ..., -0.5498,  0.2853,  0.1149],
         ...,
         [-0.2812, -0.8531,  0.6912,  ..., -0.5051,  0.4716, -0.6854],
         [-0.4429, -0.7820, -0.8055,  ...,  0.1949,  0.1081,  0.0130],
         [ 0.5570, -0.1080, -0.2412,  ...,  0.2817, -0.3996, -0.1882]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)


The same can be done providing a list of values as input

In [18]:
# Crate a batch of two.
new_input = [text] * 2
print("New input:", new_input)

# Tokenize inputs.
inputs_of_a_list = tokenizer_bert([text]*2, return_tensors='pt').to(device)
print("\n", inputs_of_a_list)

# Forward pass.
outputs_of_a_list = model_bert(**inputs_of_a_list)
# Get embeddings.
embeddings_of_a_list = outputs_of_a_list.last_hidden_state

print('\nEmbeddings of a list shape:', embeddings_of_a_list.shape, '\n')

New input: ['Hello, how are you?', 'Hello, how are you?']

 {'input_ids': tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 1029,  102],
        [ 101, 7592, 1010, 2129, 2024, 2017, 1029,  102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Embeddings of a list shape: torch.Size([2, 8, 768]) 



So far we got a list of embeddings, one per token in the sentence. How to select one embedding to represent this sentence?
- Extract the vector representation for a specific position,
- Average among all embeddings

In [19]:
# Define the sentence.
text = "This is a sentence we would like to classify."
# Tokenize it.
inputs = tokenizer_bert(
    text,
    return_tensors='pt',
    add_special_tokens=False # We introduce them later.
).to(device)

print('Input: ', inputs)

# Define token indices and attention mask.
example_input_ids = inputs['input_ids']
example_attention_mask = inputs['attention_mask']

# Forward pass.
outputs = model_bert(example_input_ids, attention_mask=example_attention_mask)

# Print the embeddings of a specific token.
pos = 3
embeddings_one_pos = outputs.last_hidden_state[:, pos, :]
print('\nEmbeddings shape:', embeddings_one_pos.shape, '\n')
print('Embeddings (first 10 el):', embeddings_one_pos[0][:10])

# Define the sencence embeddings: average last hidden state.
embeddings = outputs.last_hidden_state.mean(dim=1)
print('\nSentence embedding shape:', embeddings.shape, '\n')
print('Sentence embedding (first 10 el):', embeddings[0][:10])

Input:  {'input_ids': tensor([[ 2023,  2003,  1037,  6251,  2057,  2052,  2066,  2000, 26268,  1012]],
       device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Embeddings shape: torch.Size([1, 768]) 

Embeddings (first 10 el): tensor([ 0.6958,  0.1477, -0.0319,  0.1394,  0.3625, -0.0942, -0.0405, -0.3588,
         0.0238, -0.4620], device='cuda:0', grad_fn=<SliceBackward0>)

Sentence embedding shape: torch.Size([1, 768]) 

Sentence embedding (first 10 el): tensor([ 0.5057,  0.0430, -0.1341,  0.3907,  0.2379, -0.1453, -0.0711, -0.1567,
         0.4575, -0.6209], device='cuda:0', grad_fn=<SliceBackward0>)


Text embeddings are commonly used to define `similarity` between text sequences. They act as vectorial representations of the text itself.

The similarity between two sequences can then be calculated as the cosine similarity:
<style>
img[src="https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image.png?raw=1"],
img[src="https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image2.png?raw=1"] {
    width: 20%;
    height: auto;
}
</style>

![alt text](https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image.png?raw=1)
![alt text](https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image2.png?raw=1)


## Gemma: special tokens, tokenization, chat template, prompting


Preliminary note: to use some models from HuggingFace, you often have to accept their terms of use. For this lab, it is not required, but you are highly encouraged to signup to HuggingFace

<style>
img[src="https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image3.png?raw=1"] {
    width: 30%;
    height: auto;
}
</style>


![alt text](https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/Lab%202%3A%20Hugging%20Face%20Transformers%20%E2%80%94%20Encoder%20vs%20Decoder%E2%80%99/notebooks/images/image3.png?raw=1)

When you work inside a script/notebook, you can let the environment know that it is you by using your access token ([here](https://huggingface.co/docs/hub/security-tokens) more info), by doing so

```python
from huggingface_hub import login
login(YOUR_ACCESS_TOKEN)
```


In [20]:
# Setup: import and config
import os, json
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)

# Main paths (local cache for models)
BASE_DIR = os.path.join('lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

Transformers version: 5.0.0


In [21]:
model_path= 'unsloth/gemma-3-270m-it' #'google/gemma-3-270m-it' #  USED TO AVOID HAVING TO ACCEPT TERMS OF SERVICE FOR THE GEMMA MODEL
tokenizer = AutoTokenizer.from_pretrained(model_path, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.bfloat16,
    device_map=device
    )

print('Gemma model:', model, '\n\n')
print('Special tokens:', tokenizer.special_tokens_map, '\n\n')

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Gemma model: Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,

In [22]:
# Basic inference.
prompt = 'What are the best things to do in Padova?'

# First, you need to tokenize the prompt.
tokenized_prompt = tokenizer(
    prompt,
    return_tensors='pt',
    add_special_tokens=False
).to(device)
print('Tokenized prompt:', tokenized_prompt, '\n\n')

# Then, you can generate a response from the model.
generated_ids = model.generate(
    **tokenized_prompt,
    max_new_tokens=10,
    temperature=0.001
)
print('Generated token ids:', generated_ids, '\n\n')

# now we need to use the tokenizer to map back the generated token ids to text
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
print('Generated text:', generated_text, '\n\n')

Tokenized prompt: {'input_ids': tensor([[  3689,    659,    506,   1791,   2432,    531,    776,    528, 194486,
         236881]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')} 


Generated token ids: tensor([[  3689,    659,    506,   1791,   2432,    531,    776,    528, 194486,
         236881,  21950,  63582,  65785, 125378,  18628, 202891, 202891, 202891,
         202891, 202891]], device='cuda:0') 


Generated text: ['What are the best things to do in Padova?？” moyens”?الجبالключенияключенияключенияключенияключения'] 




This does not work properly because models have been trained with specific special tokens, that need to be inserted in the right positions.

In [23]:
prompt = 'What are the best things to do in Padova?'

# let's turn add_special_tokens to True
tokenized_prompt = tokenizer(
    prompt,
    return_tensors='pt',
    add_special_tokens=True, # <- set it true.
).to(device)

# then, you can generate a response from the model
generated_ids = model.generate(
    **tokenized_prompt,
    max_new_tokens=20,
    temperature=0.8
)
print('Generated token ids:', generated_ids, '\n\n')

# now we need to use the tokenizer to map back the generated token ids to text
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
print('Generated text:', generated_text, '\n\n')

Generated token ids: tensor([[     2,   3689,    659,    506,   1791,   2432,    531,    776,    528,
         194486, 236881,    108, 236791,  10608,    688,    568, 236791,  10608,
            688, 236768,    107, 236791,  10608,    688,    568, 236791,  10608,
            688, 236768,    107, 236791]], device='cuda:0') 


Generated text: ['<bos>What are the best things to do in Padova?\n\nPiscine (Piscine)\nPiscine (Piscine)\nP'] 




Even more, instruction-tuned models have undergone a training phase with a specific chat-like structure

In [24]:
messages = [
    {"role": "user", "content": "What are the best things to do in Padova?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<bos><start_of_turn>user
What are the best things to do in Padova?<end_of_turn>
<start_of_turn>model
Padova offers a fascinating blend of history, culture, and natural beauty. Here are some of the best things to do:
* **Explore the Roman ruins:** Hadrian's Villa, the Roman


### Parametri di generazione utili
- `max_new_tokens`: massimo numero di token generati
- `temperature`: 0.0 per output più deterministici
- `top_p`/`top_k`: campionamento nucleare
- `repetition_penalty`: penalizza ripetizioni

Nota: l'esecuzione decoder-only è più fluida con GPU. In CPU, limitare `max_new_tokens` e usare temperature basse.

### Excercise 1
You are provided a list of sentences at `lab2/data/sentences.txt`. Your need to:
- find the pair of sentences with the highest similarity
- find the sentence that has the highest overall similarity to the others. The overall similarity of a string *_s_* can be calculated as the sum of the similarities of *_s_* and all other strings.

**Hints**
function to calculate the norm of a vector: `np.linalg.norm()`

In [25]:
device = "cuda"
fn = "./lab2/data/sentences.txt"

# Store each sentence into a list.
# We also remove the double quotes.
sentences = []
with open(fn, 'r') as f:
    for line in f:
        if len(line): sentences.append(line.strip())
#print(sentences[:2])

# Tokenize.
inputs = tokenizer_bert(
    sentences,
    return_tensors="pt",
    # We add padding so that all list of tokens indices have same length.
    # In the attentino mechanism, the padding is ignored.
    padding=True
).to(device)
#print("Attention mask. Padding is ignored.\n", inputs['attention_mask'][:2])

# Get the embeddings.
output = model_bert(**inputs)
embeddings = output.last_hidden_state
print("Embeddings shape:", embeddings.shape)

# Get senteces embeddings.
sentences_emb = embeddings.mean(dim=1)
print("Sentence embeddings shape:", sentences_emb.shape)

emb = sentences_emb.cpu().detach().numpy()

def cos_sim(x, y):
    num = x.dot(y)
    den = np.linalg.norm(x) * np.linalg.norm(y)
    return num/den

max_ind = ()
max_score = -1
overall = [0] * len(sentences)
for i, x in enumerate(emb):
    for j, y in enumerate(emb):
        if j == i: continue
        score = cos_sim(x, y)
        if score > max_score:
            max_score = score
            max_ind = (i, j)
        overall[i] += score

# Print things.
print("####")
print("Max ind:", max_ind)
print("Max score:", max_score)
print("Sentences with highest similarity:")
print(sentences[max_ind[0]])
print(sentences[max_ind[1]])
print("####")
print("Sentence with max overall:")
print(sentences[np.argmax(overall)])
print("Overall max score:", max(overall))

Embeddings shape: torch.Size([25, 39, 768])
Sentence embeddings shape: torch.Size([25, 768])
####
Max ind: (19, 24)
Max score: 0.8806859
Sentences with highest similarity:
"Pilgrims visiting the Basilica of Saint Anthony find mass schedules, confession times, and lodging options nearby."
"Basilica of Saint Anthony is visited by many pilgrins. Several masses are scheduled, and they can confess."
####
Sentence with max overall:
"Padua hosts museums like MUSME (medical history) and the Museum of Pre-Cinema, offering educational exhibits linked to the university."
Overall max score: 18.50697


### Excercise 2

You are provided 3 sentences at `lab2/data/target_text.txt`. You need to construct an effective prompt to extract the mentions of people from the text. You can use the examples provided in `lab2/data/few_shot_examples.json`. You are expected to try different prompt setups, included elaborated system prompts and few-shot example prompting. \\
You need to perform inference :
- just providing the target text and a brief description of the task
- providing a proper system prompt
- providing an example on how the task needs to be performed in the user prompt
- providing an example of how the task needs to be performed as multi-turn conversation
- providing multiple examples of how the task needs to be performed as multi-turn conversation

In [26]:
# Read file.
fn = "./lab2/data/target_text.txt"
target_text = ""
with open(fn, 'r') as f:
    target_text = f.read()
print(target_text)

In 2006, the Italian national football team won the worldcup in Berlin, Germany. The team manager was Marcello Lippi.
Alfonso Suarez has been the men in charge of guiding the Spanish Governement during the transition between Franco's time and the democracy.
The fist man who got to the moon has been Armstrong, in 1969. Recently, a new NASA mission has been talking about doing it again


In [27]:
# Lets try with a simple prompt.
prompt = "I need you to extract the mentions of people from the following text." + target_text

# Tokenize.
tokenized_prompt = tokenizer(
    prompt,
    return_tensors='pt',
    add_special_tokens=True,
).to(device)

# Generate indices.
generated_ids = model.generate(
    **tokenized_prompt,
    max_new_tokens=40,
    temperature=0.1
)

# Get generated text (decod3).
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
print('Generated text:', generated_text, '\n\n')

Generated text: ["<bos>I need you to extract the mentions of people from the following text.In 2006, the Italian national football team won the worldcup in Berlin, Germany. The team manager was Marcello Lippi.\nAlfonso Suarez has been the men in charge of guiding the Spanish Governement during the transition between Franco's time and the democracy.\nThe fist man who got to the moon has been Armstrong, in 1969. Recently, a new NASA mission has been talking about doing it again.\nThe team manager of the Spanish Government has been the president of the Spanish Football Association (SFA).\nThe team manager of the Spanish Government has been the president of the Spanish Football Association ("] 




In [31]:
import json

# Load few-shot examples.
fn_examples = "./lab2/data/few_shot_examples.json"
with open(fn_examples, 'r') as f:
    few_shot_examples = json.load(f)

# Prepare messages with a few-shot example.
messages = [
    {"role": "system", "content": "Your sole task is to extract all people's names from the provided text. List them as a comma-separated string. Your response must contain only the names, with no other text, explanations, or formatting. If no names are found, return an empty string."}
]
for x in few_shot_examples[0:2]:
    messages.append({"role": "user", "content": x["input"]})
    #messages.append({"role": "assistant", "content": ", ".join(x["output"])})
    messages.append({"role": "assistant", "content": x["output"]})
messages.append({"role": "user", "content": target_text})

# Apply chat template and generate
inputs_with_example = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt"
).to(model.device)

outputs_with_example = model.generate(
    **inputs_with_example,
    max_new_tokens=40,
    temperature=0.8,
)
print(tokenizer.decode(outputs_with_example[0], skip_special_tokens=False))

<bos><start_of_turn>user
Your sole task is to extract all people's names from the provided text. List them as a comma-separated string. Your response must contain only the names, with no other text, explanations, or formatting. If no names are found, return an empty string.

Barack Obama met Angela Merkel in Berlin. in 2014, to discuss about the economic situation.<end_of_turn>
<start_of_turn>model
<end_of_turn>
<start_of_turn>user
Maria Rossi called Luca yesterday, as he wanted to book a table at the restaurant.<end_of_turn>
<start_of_turn>model
<end_of_turn>
<start_of_turn>user
In 2006, the Italian national football team won the worldcup in Berlin, Germany. The team manager was Marcello Lippi.
Alfonso Suarez has been the men in charge of guiding the Spanish Governement during the transition between Franco's time and the democracy.
The fist man who got to the moon has been Armstrong, in 1969. Recently, a new NASA mission has been talking about doing it again<end_of_turn>
<start_of_tur